# attoDRY2100 Magnet Control — **Using attocube-provided Python files**

This notebook imports your attocube Python package (the folder that contains `__init__.py`, `attoDry2100.py`, `magnet.py`, etc.) and uses its API to ramp/sweep the magnet with settle logic suitable for MCD measurements. No virtual environment required.

**How to run in VS Code:**
1. Select your **system Python 3.11** interpreter.
2. Set `SDK_DIR` and `HOST` below.
3. Run the cells top→bottom. Start with a small test ramp before full sweeps.


In [1]:
# --- Configuration ---
from pathlib import Path
SDK_DIR = Path(r"D:\Insturment control v3\CRYO2100")  # <-- CHANGE to your local folder if running on your PC
HOST = "192.168.1.1"          # eNSPIRE/attoDRY host IP
CHANNEL = 0                     # single-axis channel index
POLL_HZ = 10

LIMITS = {
    "field_T_max": 9.0,
    "ramp_T_per_min_max": 0.3,
    "safe_temp_K_min": 4.2,
    "safe_temp_K_max": 320.0,
}
SETTLE = {
    "tol_T": 1e-3,            # 1 mT absolute
    "rel_tol_fraction": 2e-4, # 0.02% of |B|
    "slope_T_per_s": 5e-4,    # dB/dt threshold
    "window_s": 1.0,
    "hold_s": 1.0,
}


In [2]:
# --- Import the attocube package from SDK_DIR ---
import sys, importlib.util
def import_package_from_path(package_name: str, package_path: Path):
    init_file = package_path / "__init__.py"
    if not init_file.exists():
        raise FileNotFoundError(f"__init__.py not found in {package_path}")
    spec = importlib.util.spec_from_file_location(package_name, init_file, submodule_search_locations=[str(package_path)])
    module = importlib.util.module_from_spec(spec)
    sys.modules[package_name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

SDK = import_package_from_path("attocube_sdk", SDK_DIR)
from attocube_sdk import Device  # provided by your __init__.py
print("Loaded attocube SDK from:", SDK_DIR)


Loaded attocube SDK from: D:\Insturment control v3\CRYO2100


In [3]:
# --- Connect to device and get magnet interface ---
dev = Device(HOST)
magnet = dev.magnet
print("Connected to:", HOST)


Connected to: 192.168.1.1


In [4]:
# --- Safety + settle helpers ---
import time
from collections import deque

class SafetySupervisor:
    def __init__(self, limits: dict):
        self.limits = limits
    def precheck(self, target_T: float, rate_T_per_min: float, temp_K: float):
        assert abs(target_T) <= self.limits["field_T_max"], "field limit"
        assert 0 < rate_T_per_min <= self.limits["ramp_T_per_min_max"], "ramp limit"
        assert self.limits["safe_temp_K_min"] <= temp_K <= self.limits["safe_temp_K_max"], "temp not safe"

class Settler:
    def __init__(self, read_field, tol_T=1e-3, rel_tol=2e-4, slope_T_per_s=5e-4, window_s=1.0, hold_s=1.0, poll_hz=10):
        self.read_field = read_field
        self.tol_T = tol_T
        self.rel_tol = rel_tol
        self.slope = slope_T_per_s
        self.win = window_s
        self.hold = hold_s
        self.dt = 1.0 / poll_hz
        self.buf = deque(maxlen=int(self.win / self.dt) + 2)
    def _tol(self, target):
        return max(self.tol_T, abs(target) * self.rel_tol)
    def wait_stable(self, target_T):
        ok_since = None
        while True:
            time.sleep(self.dt)
            t = time.time(); B = self.read_field()
            self.buf.append((t, B))
            if len(self.buf) < 2:
                continue
            t0, B0 = self.buf[0]; t1, B1 = self.buf[-1]
            dBdt = (B1 - B0) / max(1e-6, (t1 - t0))
            in_tol = abs(B1 - target_T) <= self._tol(target_T)
            flat = abs(dBdt) <= self.slope
            if in_tol and flat:
                ok_since = ok_since or t1
                if (t1 - ok_since) >= self.hold:
                    return B1
            else:
                ok_since = None


In [5]:
from pathlib import Path
import csv, os
class LiveCSV:
    def __init__(self, path, header, meta=None, fsync=True):
        self.path = Path(path)
        self.fsync = fsync
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if not self.path.exists() or self.path.stat().st_size == 0:
            with open(self.path, "w", newline="") as f:
                if meta is not None:
                    f.write("# " + json.dumps(meta) + "\n")
                csv.writer(f).writerow(header)
                f.flush()
                if self.fsync: os.fsync(f.fileno())
    def write_row(self, row):
        with open(self.path, "a", newline="") as f:
            csv.writer(f).writerow(row)
            f.flush()
            if self.fsync: os.fsync(f.fileno())



In [6]:
# --- Magnet control using attocube API ---
safety = SafetySupervisor(LIMITS)
settler = Settler(lambda: float(magnet.getH(CHANNEL)),
                  tol_T=SETTLE["tol_T"], rel_tol=SETTLE["rel_tol_fraction"],
                  slope_T_per_s=SETTLE["slope_T_per_s"], window_s=SETTLE["window_s"],
                  hold_s=SETTLE["hold_s"], poll_hz=POLL_HZ)

def ramp_to(target_T: float, rate_T_per_min: float):
    temp_K = float(magnet.getTemperature())
    safety.precheck(target_T, rate_T_per_min, temp_K)
    # If your system uses multiple rate profiles, adjust (index, range) accordingly
    magnet.setRampRate(CHANNEL, 0, 0.0, float(rate_T_per_min))
    magnet.setHSetPoint(CHANNEL, float(target_T))
    magnet.startFieldControl(CHANNEL)
    B_stable = settler.wait_stable(target_T)
    return B_stable

# Simple settle criteria (reuse if you already have one)
import time
from collections import deque

SETTLE = dict(tol_T=1e-3, rel_tol_fraction=2e-4, slope_T_per_s=5e-4, window_s=1.0, hold_s=1.0)

def print_states(where=""):
    print(f"[{where}] driven={bool(magnet.getDrivenMode(CHANNEL))}  "
          f"persist={bool(magnet.getPersistentMode(CHANNEL))}  "
          f"leads_hot={bool(magnet.getLeadsHot())}  "
          f"field_ctrl={bool(magnet.getFieldControl(CHANNEL))}  "
          f"B={float(magnet.getH(CHANNEL)):.6f} T  Tmag={float(magnet.getTemperature()):.2f} K")


import time
from collections import deque

def _wait_stable_field(target_T):
    buf = deque(maxlen=int(SETTLE["window_s"]*POLL_HZ)+2)
    dt = 1.0 / POLL_HZ
    ok_since = None
    def _tol(): return max(SETTLE["tol_T"], abs(target_T)*SETTLE["rel_tol_fraction"])
    while True:
        time.sleep(dt)
        t = time.time(); B = float(magnet.getH(CHANNEL))
        buf.append((t, B))
        if len(buf) < 2: 
            continue
        t0,B0 = buf[0]; t1,B1 = buf[-1]
        dBdt = (B1 - B0)/max(1e-6,(t1 - t0))
        in_tol = abs(B1 - target_T) <= _tol()
        flat   = abs(dBdt) <= SETTLE["slope_T_per_s"]
        if in_tol and flat:
            ok_since = ok_since or t1
            if (t1 - ok_since) >= SETTLE["hold_s"]:
                return B1
        else:
            ok_since = None


def ramp_to_using_current_rate(B_target_T, *, settle=True):
    """
    Do NOT change persistent/driven mode or ramp preset.
    Let the controller handle heater/autoswitching internally.
    """
    print_states("pre-ramp")
    magnet.setHSetPoint(CHANNEL, float(B_target_T))
    magnet.startFieldControl(CHANNEL)  # controller may auto-heat/cool under the hood
    B_final = _wait_stable_field(float(B_target_T)) if settle else float(magnet.getH(CHANNEL))
    print_states("post-ramp")
    return B_final

In [7]:
def steps_inclusive(start, stop, step):
    sgn = 1 if stop >= start else -1
    step = abs(step) * sgn
    x = start
    while (sgn > 0 and x <= stop + 1e-12) or (sgn < 0 and x >= stop - 1e-12):
        yield round(x, 12); x += step

In [8]:
import pyvisa, iv_automation, lf6_automation, spectral_experiments
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import winsound, time, os
rm = pyvisa.ResourceManager()
print(rm.list_resources())
plt.rcParams['figure.raise_window'] = False

('ASRL1::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL7::INSTR', 'ASRL8::INSTR', 'ASRL9::INSTR', 'GPIB0::2::INSTR', 'GPIB0::3::INSTR')


In [9]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('ASRL1::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL7::INSTR', 'ASRL8::INSTR', 'ASRL9::INSTR', 'GPIB0::2::INSTR', 'GPIB0::3::INSTR')


In [10]:
# change address
keithley1 = iv_automation.KeithControl('GPIB0::02::INSTR', 'GPIB26', 'Vtg', rm)
keithley1.set_volt_step(curr_compliance=1E-8, delay=0.05, volt_compliance=50)
# change address
keithley2 = iv_automation.KeithControl('GPIB0::03::INSTR', 'GPIB23', 'Vbg', rm)
keithley2.set_volt_step(curr_compliance=1E-8, delay=0.05, volt_compliance=50)
instrument_list = [keithley1, keithley2]
iv = iv_automation.IVSetup(instrument_list)

#x_goto(self, x_name, target, delta, delay)
iv.x_goto('Vtg', 0, 0.1, 0.1)
iv.x_goto('Vbg', 0, 0.1, 0.1)
iv.report_status()

KEITHLEY INSTRUMENTS INC.,MODEL 2400,4502039,D02 Jan 20 2021 10:18:49/B01  /W/N
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-06
SOUR:DEL 0.100
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 20
TRIG:COUN 1
:OUTP ON
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-08
SOUR:DEL 0.050
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 50
TRIG:COUN 1
:OUTP ON
KEITHLEY INSTRUMENTS INC.,MODEL 2400,1045870,C32   Oct  4 2010 14:20:11/A02  /K/H
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-06
SOUR:DEL 0.100
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 20
TRIG:COUN 1
:OUTP ON
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-08
SOUR:DEL 0.050
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 50
TRIG:COUN 1
:OUTP ON
x_channel Vtg: value: 0.0
x_channel Vbg: value: 0.0
y_channel measured_Vtg: value: 0.0
y_channel Vtg_leakage: value: -2.746787e-11

In [12]:
lf6 = lf6_automation.LF6Setup()
lf6.print_saved_experiments()

My Saved Experiments:
	2100
	attodry1000 new
	attodry1000
	cxt
	EMCCD
	Experiment1
	Experiment2
	Experiment3
	Modulation
	PL_Lei
	ref
	test


In [13]:
# --- Connect to attoDRY / eNSPIRE ---
from attocube_sdk import Device

def ensure_connected():
    global dev, magnet
    try:
        if 'dev' in globals() and getattr(dev, "is_open", False):
            return
    except Exception:
        pass
    # (Re)create and connect
    try:
        dev = Device(HOST)      # some SDKs take host in ctor
        dev.connect()
    except TypeError:
        dev = Device()          # others want host in connect()
        dev.connect(HOST)
    magnet = dev.magnet
    print("Connected to", HOST, "| open =", getattr(dev, "is_open", True))

# ensure_connected()

In [14]:
def set_gate(name: str, target: float, step: float = 0.1, delay: float = 0.1):
    """Use your API: iv.x_goto(name, target, step, delay)."""
    iv.x_goto(name, float(target), float(step), float(delay))

def set_gates_once(Vbg: float | None = None, Vtg: float | None = None,
                   step_bg: float = 0.1, delay_bg: float = 0.1,
                   step_tg: float = 0.1, delay_tg: float = 0.1):
    if Vbg is not None: set_gate('Vbg', Vbg, step_bg, delay_bg)
    if Vtg is not None: set_gate('Vtg', Vtg, step_tg, delay_tg)
    return Vbg, Vtg


In [15]:
# Run this once before your sweep
def configure_lf6(*, exp_time=None, frame_to_combine=None, center=None, verbose=True):
    """
    Apply LF6 settings once at the start.
    NOTE: Use the units your LF6 driver expects (often ms for exposure).
    """
    changed = {}
    if exp_time is not None:
        lf6.change_expose_time(exp_time)
        changed["exp_time"] = exp_time
    if frame_to_combine is not None:
        lf6.change_frame_to_combine(frame_to_combine)
        changed["frame_to_combine"] = frame_to_combine
    if center is not None:
        lf6.change_spectra_center(center)
        changed["center"] = center
    if verbose and changed:
        print("LF6 configured:", changed)
    return changed


In [16]:
import re, time
from pathlib import Path

def _sanitize(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._+-]", "_", s)

def _suffix_from_params(tg_bg_ratio, Vbg, Vtg, lf6_center, start_T, stop_T, bidirectional, angles_deg):
    gate = f"Vbg={Vbg}_Vtg={Vtg}V"
    Efield = f"E={np.round(tg_bg_ratio*Vtg-Vbg,3)}"
    Doping = f"D={np.round(tg_bg_ratio*Vtg+Vbg,3)}"

    lf6_center = f"{lf6_center}nmc"
    Brange = f"B{start_T:+.1f}Tto{stop_T:+.1f}T"
    mode   = "bi" if bidirectional else "uni"
    angblk = f"inhalf{'_'.join(str(round(a,2)) for a in angles_deg)}" if angles_deg else None
    parts  = [gate,Efield,Doping, lf6_center, Brange, mode] + ([angblk] if angblk else [])
    return "_".join(parts)

def resolve_csv_path(
    base_dir: Path,
    filename_base: str | None,
    *,
    start_T: float, stop_T: float,
    bidirectional: bool,
    lf6_center: str,
    tg_bg_ratio,Vbg,Vtg,
    angles_deg: list[float] | None,
    timestamp: bool = True,
    exact: bool = False,
) -> Path:
    """
    If exact=True:
        - if filename_base endswith .csv -> use exactly (in base_dir if no folder given)
        - else append .csv
    Else:
        - build: <base>_<B...>_<bi|uni>[_inhalf..]_[YYYYmmdd_HHMMSS].csv
    """
    base_dir = Path(base_dir)
    if not filename_base:
        filename_base = "run"
    name = _sanitize(filename_base)

    if exact:
        if name.lower().endswith(".csv"):
            fname = name
        else:
            fname = name + ".csv"
        p = Path(fname)
        return p if p.parent != Path("") else base_dir / p.name

    suffix = _suffix_from_params(tg_bg_ratio,Vbg,Vtg,lf6_center, start_T, stop_T, bidirectional, angles_deg)
    ts = time.strftime("%Y%m%d_%H%M%S", time.localtime()) if timestamp else None
    parts = ([ts] if ts else []) + [name, suffix]  
    return base_dir / ("_".join(parts) + ".csv")

In [17]:
from backend.instruments import RS232EP300
ep300 = RS232EP300.RS232EP300(address='ASRL5::INSTR', axes=[1])
ep300.connect()
# ep300.set_speed(2, 20)
ep300.set_speed(1, 20)

def wp_move_to(angle_deg: float, wait_s: float = 0.5, settle_s: float = 0.3):
    """Move waveplate to angle (deg). Best-effort wait."""
    if ep300 is None:
        raise RuntimeError("Waveplate rotator not initialized")
    ep300.set_position(1,float(angle_deg))
    # simple wait; use device status if available in your API
    import time
    time.sleep(wait_s + settle_s)

In [61]:
wp_move_to(158-90
)


In [433]:
ep300.close()

In [62]:
DEV_NAME = 'YZD300_MCD'

DATA_DIR     = Path(rf"./{DEV_NAME}");        
DATA_DIR.mkdir(exist_ok=True, parents=True)
SPECTRA_DIR  = DATA_DIR / "spectra";  SPECTRA_DIR.mkdir(exist_ok=True, parents=True)
spectral_experiments.USER_FOLDER = rf"D:\Insturment control v3_1\{DEV_NAME}\initial data"

In [22]:
# -- Fast, bounded wait to get near a target without full settle (used only for pre-position) --
import time, csv
from collections import deque

POLL_HZ = 10
SETTLE = dict(tol_T=1e-3, rel_tol_fraction=2e-4, slope_T_per_s=5e-4, window_s=1.0, hold_s=1.0)


def emit_event(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def _abs_tol(T): 
    return max(1e-3, abs(T) * 2e-4)  # 1 mT or 0.02%·|T|

def _is_idle(state) -> bool:
    try:
        return "idle" in str(state).strip().lower()
    except Exception:
        return False

def wait_idle_close(target_T: float, *, timeout_s: float = 180.0, poll_s: float = 0.1):
    """
    Wait until controller says 'idle' **and** measured B is within absolute tolerance of target_T.
    No dB/dt or slope used.
    """
    tol = _abs_tol(target_T)
    t0 = time.time()
    while True:
        state = magnet.getHState(CHANNEL)
        B = float(magnet.getH(CHANNEL))
        if _is_idle(state) and abs(B - target_T) <= tol:
            return B
        if time.time() - t0 >= timeout_s:
            raise TimeoutError(f"Idle wait timed out at B={B:.4f} T, state={state!r}")
        time.sleep(poll_s)

def preposition_to_start_gate(start_T: float, *, gate_T: float | None = None,
                              max_wait_s: float = 300.0, poll_s: float = 0.1,
                              consecutive: int = 5, pause_s: float = 0.5):
    """Move to start_T and wait until we read `consecutive` values within |B-start_T| <= gate."""
    tol = gate_T if gate_T is not None else _abs_tol(start_T)
    magnet.setHSetPoint(CHANNEL, float(start_T))
    magnet.startFieldControl(CHANNEL)
    emit_event(f"Pre-positioning to {start_T:+.4f} T (gate ±{tol:.4f} T) …")

    t0 = time.time(); ok = 0; B = float(magnet.getH(CHANNEL))
    while True:
        time.sleep(poll_s)
        B = float(magnet.getH(CHANNEL))
        if abs(B - start_T) <= tol:
            ok += 1
            if ok >= consecutive:
                emit_event(f"Reached start gate: B≈{B:+.4f} T")
                break
        else:
            ok = 0
        if time.time() - t0 >= max_wait_s:
            emit_event(f"WARNING: start gate timeout (B≈{B:+.4f} T). Proceeding.")
            break
    if pause_s > 0: time.sleep(pause_s)
    return B



In [23]:
def run_continuous_leg_gate(*, start_T: float, stop_T: float,
                            wls_len: int, angles_deg: list[float] | None,
                            start_gate_T: float | None = None,
                            label: str = "forward",
                            turn_pause_s: float = 0.5,
                            live: LiveCSV | None = None):
    """
    1) Pre-position to start_T using absolute gate (no Idle).
    2) Set ONE setpoint to stop_T and acquire spectra back-to-back while ramping.
    3) Stop this leg when we cross into |B-stop_T| <= tol (no Idle).
    """
    if live is None: raise RuntimeError("LiveCSV appender required")

    # 1) reach start gate (not logged)
    preposition_to_start_gate(start_T, gate_T=start_gate_T, pause_s=0.5)

    # 2) arm single ramp to stop_T
    emit_event(f"Start {'backward ' if label.startswith('back') else ''}ramping to {stop_T:+.4f} T")
    magnet.setHSetPoint(CHANNEL, float(stop_T))
    magnet.startFieldControl(CHANNEL)

    sgn = 1 if stop_T >= start_T else -1
    tol_stop = _abs_tol(stop_T)
    angles = angles_deg if angles_deg else [None]

    while True:
        B_now = float(magnet.getH(CHANNEL))
        # End of leg when we enter stop gate (don’t require Idle)
        if (sgn > 0 and B_now >= stop_T - tol_stop) or (sgn < 0 and B_now <= stop_T + tol_stop):
            emit_event(f"Stop ramping ({label} end): B≈{B_now:+.4f} T")
            break

        for ang in angles:
            if ang is not None:
                wp_move_to(ang)

            # acquire while ramping; log mid-acquisition field
            B0 = float(magnet.getH(CHANNEL))
            spec = lf6.acquire()            # ~2 s
            B1 = float(magnet.getH(CHANNEL))
            B_meas = 0.5 * (B0 + B1)

            arr = np.asarray(spec, dtype=int).ravel()
            if arr.size < wls_len:  arr = np.pad(arr, (0, wls_len - arr.size), constant_values=np.nan)
            elif arr.size > wls_len: arr = arr[:wls_len]

            live.write_row([B_meas, (float(ang) if ang is not None else np.nan), *arr.tolist()])


In [24]:
def run_B_points_gate(*,
                      B_points: list[float] | np.ndarray,
                      wls_len: int,
                      angles_deg: list[float] | None,
                      live: LiveCSV,
                      settle_pause_s: float = 0.5):
    """
    Discrete B-point mode.

    Sweep through an explicit list of B points in the GIVEN ORDER:
        e.g. [-2.0, -1.0, 0.0, 1.0, 2.0]

    For each B_target:
      - Set magnet setpoint to B_target
      - Wait until |B_now - B_target| <= tol
      - Optional extra settling time
      - Loop over all waveplate angles and acquire spectra
    """
    import time
    B_arr = np.asarray(B_points, float).ravel()
    if B_arr.size == 0:
        raise ValueError("B_points must be a non-empty list/array.")

    if live is None:
        raise RuntimeError("LiveCSV appender required")

    angles = angles_deg if angles_deg else [None]

    # Use the first point as initial pre-position
    B_first = float(B_arr[0])
    preposition_to_start_gate(B_first, gate_T=None, pause_s=settle_pause_s)

    # Now walk through the list as given
    for i, B_target in enumerate(B_arr, start=1):
        B_target = float(B_target)
        emit_event(f"Set field to {B_target:+.4f} T (B point {i}/{len(B_arr)})")

        if i != 1:
            magnet.setHSetPoint(CHANNEL, B_target)
            magnet.startFieldControl(CHANNEL)

            tol = _abs_tol(B_target)
            while True:
                B_now = float(magnet.getH(CHANNEL))
                if abs(B_now - B_target) <= tol:
                    break
                time.sleep(0.2)

            if settle_pause_s > 0:
                time.sleep(settle_pause_s)

        # At this fixed field, measure all waveplate angles
        for ang in angles:
            if ang is not None:
                wp_move_to(ang)

            B0 = float(magnet.getH(CHANNEL))
            spec = lf6.acquire()
            B1 = float(magnet.getH(CHANNEL))
            B_meas = 0.5 * (B0 + B1)

            arr = np.asarray(spec, dtype=int).ravel()
            if arr.size < wls_len:
                arr = np.pad(arr, (0, wls_len - arr.size), constant_values=np.nan)
            elif arr.size > wls_len:
                arr = arr[:wls_len]

            live.write_row([
                B_meas,
                (float(ang) if ang is not None else np.nan),
                *arr.tolist()
            ])


In [65]:
# def sweep_fixed_gate_continuous_to_single_csv(
#     *,
#     filename_base: str | None,
#     start_T: float, stop_T: float,
#     bidirectional: bool = False,
#     tg_bg_ratio: float = 1,
#     Vbg: float | None = None, Vtg: float | None = None,
#     waveplate_angles_deg: list[float] | None = None,
#     return_angle_deg: float | None = None,
#     lf6_exp_time=None, lf6_frames=None, lf6_center=None,
#     out_csv: str | None = None, filename_exact: bool = False,
#     B_points: list[float] | np.ndarray | None = None,   # NEW
# ):
#     ensure_connected()

#     # 1) Apply gates + LF6 settings once
#     set_gates_once(Vbg, Vtg)
#     configure_lf6(exp_time=lf6_exp_time,
#                   frame_to_combine=lf6_frames,
#                   center=lf6_center,
#                   verbose=True)

#     # 2) Wavelength calibration
#     wls = np.asarray(lf6.get_wavelength_calibration(), dtype=float).ravel()
#     n_wls = int(wls.size)

#     # --- normalize B_points to a 1D array, or None ---
#     if B_points is not None:
#         B_points_arr = np.asarray(B_points, float).ravel()
#         if B_points_arr.size == 0:
#             B_points_arr = None
#     else:
#         B_points_arr = None

#     # For naming, use B_points range if provided
#     if B_points_arr is not None:
#         B_min = float(B_points_arr.min())
#         B_max = float(B_points_arr.max())
#         start_T_for_name = B_min
#         stop_T_for_name  = B_max
#     else:
#         start_T_for_name = start_T
#         stop_T_for_name  = stop_T

#     # 3) Resolve CSV path
#     csv_path = Path(out_csv) if out_csv else resolve_csv_path(
#         SPECTRA_DIR, filename_base,
#         start_T=start_T_for_name, stop_T=stop_T_for_name,
#         bidirectional=bidirectional,
#         tg_bg_ratio=tg_bg_ratio, Vbg=Vbg, Vtg=Vtg,
#         lf6_center=lf6_center,
#         angles_deg=waveplate_angles_deg,
#         timestamp=True,
#         exact=filename_exact
#     )
#     csv_path.parent.mkdir(parents=True, exist_ok=True)

#     # 4) Write header & run sweep
#     with open(csv_path, "w", newline="") as f:
#         header = ["B_T", "angle_deg", *wls.tolist()]
#         live = LiveCSV(csv_path, header, fsync=True)

#         if B_points_arr is not None:
#             # --- discrete B-point mode (e.g. np.linspace(-2,2,21)) ---
#             run_B_points_gate(
#                 B_points=B_points_arr,
#                 wls_len=n_wls,
#                 angles_deg=waveplate_angles_deg,
#                 live=live,
#                 settle_pause_s=0.5,
#             )

#             if bidirectional:
#                 # Reverse and skip the first element to avoid duplicate endpoint
#                 if B_points_arr.size > 1:
#                     B_points_rev = B_points_arr[::-1][1:]
#                     run_B_points_gate(
#                         B_points=B_points_rev,
#                         wls_len=n_wls,
#                         angles_deg=waveplate_angles_deg,
#                         live=live,
#                         settle_pause_s=0.5,
#                     )

#         else:
#             # --- original continuous mode ---
#             run_continuous_leg_gate(
#                 start_T=start_T,
#                 stop_T=stop_T,
#                 wls_len=n_wls,
#                 angles_deg=waveplate_angles_deg,
#                 start_gate_T=None,
#                 label="forward",
#                 turn_pause_s=0.5,
#                 live=live,
#             )

#             if bidirectional:
#                 run_continuous_leg_gate(
#                     start_T=stop_T,
#                     stop_T=start_T,
#                     wls_len=n_wls,
#                     angles_deg=waveplate_angles_deg,
#                     start_gate_T=None,
#                     label="backward",
#                     turn_pause_s=0.0,
#                     live=live,
#                 )

#     if return_angle_deg is not None and 'ep300' in globals() and ep300 is not None:
#         wp_move_to(return_angle_deg)

#     print("Saved:", csv_path)
#     return str(csv_path)


In [66]:
# --- Top-level continuous sweep to ONE CSV (supports hysteresis & two angles) ---
def sweep_fixed_gate_continuous_to_single_csv(
    *,
    filename_base: str | None,     # e.g., "YZD320_1.67Kp1" (single string you edit)
    start_T: float, stop_T: float,
    bidirectional: bool = False,   # True → append return leg in same CSV
    tg_bg_ratio: float =1,
    Vbg: float | None = None, Vtg: float | None = None,
    waveplate_angles_deg: list[float] | None = None,  # e.g., [84, 42]
    return_angle_deg: float | None = None,
    lf6_exp_time=None, lf6_frames=None, lf6_center=None,
    out_csv: str | None = None, filename_exact: bool = False,
):
    ensure_connected()
    # 1) Apply gates + LF6 settings once
    set_gates_once(Vbg, Vtg)
    configure_lf6(exp_time=lf6_exp_time, frame_to_combine=lf6_frames, center=lf6_center, verbose=True)

    # 2) Wavelengths (for header)
    wls = np.asarray(lf6.get_wavelength_calibration(), dtype=float).ravel()
    n_wls = int(wls.size)

    # 3) Resolve CSV path from your base name
    csv_path = Path(out_csv) if out_csv else resolve_csv_path(
        SPECTRA_DIR, filename_base,
        start_T=start_T, stop_T=stop_T, bidirectional=bidirectional,
        tg_bg_ratio=tg_bg_ratio, Vbg=Vbg,Vtg=Vtg,lf6_center = lf6_center,
        angles_deg=waveplate_angles_deg, timestamp=True, exact=filename_exact
    )
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    # 4) Write header + run legs
    with open(csv_path, "w", newline="") as f:
        
        # Row 0: B_T, angle_deg, λ...
        header = ["B_T", "angle_deg", *wls.tolist()]
        live = LiveCSV(csv_path, header, fsync=True)


        # Forward leg: start_T -> stop_T
        run_continuous_leg_gate(start_T=start_T, stop_T=stop_T,
                                wls_len=n_wls, angles_deg=waveplate_angles_deg,
                                start_gate_T=None,     # or e.g. 0.01 for ±10 mT start gate
                                label="forward",
                                turn_pause_s=0.5, live=live)

        # Backward leg (optional): stop_T -> start_T
        if bidirectional:
            run_continuous_leg_gate(start_T=stop_T, stop_T=start_T,
                                    wls_len=n_wls, angles_deg=waveplate_angles_deg,
                                    start_gate_T=None,
                                    label="backward",
                                    turn_pause_s=0.0, live=live)
            
    # Optional: park waveplate
    if return_angle_deg is not None and 'ep300' in globals() and ep300 is not None:
        wp_move_to(return_angle_deg)

    print("Saved:", csv_path)
    return str(csv_path)


In [27]:
import json
def _san(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._+-]", "_", str(s))

def make_R0_filename(
    base_dir,
    filename_base,
    *,
    angle_deg,
    Vbg,
    Vtg,
    center_nm=None,         # e.g. 550
    exp_ms=None,            # e.g. 200 (ms)
    frames=None,            # e.g. 10
    timestamp=True
):
    base_dir = Path(base_dir)
    ang  = f"ang{int(round(float(angle_deg)))}" if angle_deg is not None else "angNA"
    cen  = f"c{int(round(float(center_nm)))}nm" if center_nm is not None else "cNA"
    expf = (
        f"exp{int(round(float(exp_ms)))}x{int(frames)}"
        if (exp_ms is not None and frames is not None) else "expNA"
    )
    vbg  = f"Vbg{float(Vbg):+0.3f}" if Vbg is not None else "VbgNA"
    vtg  = f"Vtg{float(Vtg):+0.3f}" if Vtg is not None else "VtgNA"
    parts = [_san(filename_base), "B0T", ang, cen, expf, vbg, vtg, "R0"]
    if timestamp:
        parts.append(time.strftime("%Y%m%d_%H%M%S", time.localtime()))
    return base_dir / ("_".join(parts) + ".csv")


def save_R0_for_angles(
    *,
    base_dir="./data",
    filename_base: str,          # e.g. "YZD320_1.67Kp1"
    angles_deg: list[float],     # e.g. [82, 42]
    Vbg: float, Vtg: float,
    gate_T: float = 0.001,       # ±1 mT window for 0 T
    repeats: int = 1,            # rows per angle
    average: bool = False,       # True -> single averaged row per angle
    lf6_exp_time=None, lf6_frames=None, lf6_center=None,
    timestamp: bool = True
) -> list[str]:
    """
    For each angle: set gates (once), move waveplate, ensure |B|<=gate_T, acquire R0,
    and write a CSV with numeric wavelength header and R(λ) rows.
    Returns list of CSV paths.
    """
    # 1) Gates + LF6 (once)
    set_gates_once(Vbg, Vtg)
    lf6_changed = configure_lf6(exp_time=lf6_exp_time, frame_to_combine=lf6_frames, center=lf6_center, verbose=True)


    # 2) Wavelengths AFTER any LF6 center change
    wls = np.asarray(lf6.get_wavelength_calibration(), dtype=float).ravel()
    if wls.size == 0:
        raise RuntimeError("LF6 wavelength calibration returned 0 elements.")
    n_w = int(wls.size)

    outputs = []

    for ang in angles_deg:
        # 3) Waveplate
        wp_move_to(ang)
        time.sleep(2)

        # # 4) Ensure 0 T gate before measuring (quick check each angle)
        # _ = goto_zero_field(gate_T=gate_T)

        # 5) Acquire one or more spectra
        rows = []
        for k in range(int(repeats)):
            spec = lf6.acquire()
            arr = np.asarray(spec, dtype=int).ravel()
            if arr.size < n_w:  arr = np.pad(arr, (0, n_w - arr.size), constant_values=np.nan)
            elif arr.size > n_w: arr = arr[:n_w]
            rows.append(arr)

        if average and len(rows) > 1:
            body = [np.nanmean(np.vstack(rows), axis=0)]
        else:
            body = rows

        # 6) Build filename with angle + gate voltages and save CSV (numeric header only)
        center_nm = lf6_center
        exp_ms    = lf6_exp_time
        frames    = lf6_frames

        # Build filename that includes angle + LF6 + gates
        out_csv = make_R0_filename(
            base_dir, filename_base,
            angle_deg=ang, Vbg=Vbg, Vtg=Vtg,
            center_nm=center_nm, exp_ms=exp_ms, frames=frames,
            timestamp=timestamp
        )
        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        with open(out_csv, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(wls.tolist())                  # numeric header
            for r in body:
                w.writerow(np.asarray(r, dtype=float).tolist())

        # Optional: write sidecar metadata (keeps CSV “simple & safest”)
        meta = {
            "ts": time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime()),
            "angle_deg": float(ang),
            "Vbg": float(Vbg), "Vtg": float(Vtg),
            "gate_T": float(gate_T),
            "repeats": int(repeats), "average": bool(average),
            "lf6": lf6_changed
        }
        Path(out_csv).with_suffix(".json").write_text(json.dumps(meta, indent=2))
        print(f"Saved R0: {out_csv}")
        outputs.append(str(out_csv))

    return outputs


## Quick tests (use safe small fields first)


In [28]:
# Connect the attocube Device before using magnet.*
# Works with both common signatures:
try:
    dev.connect()          # if Device(HOST) was used at construction time
except TypeError:
    dev.connect(HOST)      # if Device() needs the host passed here

# optional: quick sanity checks
try:
    print("Connected:", getattr(dev, "is_open", True))
except Exception:
    pass

magnet = dev.magnet  # (re)bind after connect to be safe

Connected: True


In [29]:
# Read basic telemetry
print("Temp [K]:", float(magnet.getTemperature()))
print("Field [T]:", float(magnet.getH(CHANNEL)))
print("Driven mode:", bool(magnet.getDrivenMode(CHANNEL)))
print("Persistent:", bool(magnet.getPersistentMode(CHANNEL)))


Temp [K]: 4.7615
Field [T]: 8.9992
Driven mode: False
Persistent: True


In [ ]:
import sympy as sp

tg_start, bg_start, tg_stop, bg_stop = sp.symbols('tg_start bg_start tg_stop bg_stop')
tg_bg_ratio=0.95
Efields=[-5]
bg_start_vals = []
tg_start_vals = []
for Efield in Efields:
    doping = 8.68
    # 16.2
    #n=0   0
    #n=1/3 3.68
    #n=2/3 6.356
    #n=1   9.0
    # two equations
    ## -10, 18.8
    eq1 = sp.Eq(tg_bg_ratio*tg_start - bg_start, Efield)
    eq2 = sp.Eq(tg_bg_ratio*tg_start + bg_start, doping)


    # solve for tg, bg
    sol1 = sp.solve((eq1, eq2), (tg_start, bg_start))
    # print(sol1)
    # print(sol2)
    # print({k: float(v) for k, v in sol1.items()})
    bg_start_val = float(sol1[bg_start])
    tg_start_val = float(sol1[tg_start])
    bg_start_vals.append(round(bg_start_val,2))
    tg_start_vals.append(round(tg_start_val,2))
print("bg_start_vals:", bg_start_vals)
print("tg_start_vals:", tg_start_vals)

bg_start_vals: [-7.0]
tg_start_vals: [18.95]


In [430]:
Vbg = 0
Vtg = 0
set_gates_once(Vbg, Vtg)


variable: ['Vbg']
start: [11.84]
end: [0.]
steps: 119
variable: ['Vtg']
start: [-3.33]
end: [0.]
steps: 34


(0, 0)

In [429]:
# sweep up and down for measure R+ and R-

# Vbgs = [-3,   -0.5,  2   ,4.5  ,7  ,  9.5,    12,  14.5,  17,   19.5]
# Vtgs = [12.63, 10,  7.37 ,4.77, 2.11, -0.53, -3.16,-5.79, -8.42,-11.05]


Vbgs = [-18.45, -15.91, -12.05, -13.25, -9.5, -7.00, -10.83, -5.74, -1.85, -8.28, -4.43, 8.25, 5.75 , 3.250, 0.75, 6.84, 9.34, 11.34, 11.84  ]
Vtgs = [27.95, 25.35, 29.42,  22.79 , 26.84,  24.21, 20.18, 15.02, 19.11, 17.61, 21.65,  8.68, 11.32, 13.95, 16.58, 1.94, -0.69, -2.8, -3.33 ]

# Vbgs=[-8.28]
# Vtgs=[17.61]
tg_bg_ratio = 0.95

for Vbg,Vtg in zip(Vbgs,Vtgs):
    csv_path = sweep_fixed_gate_continuous_to_single_csv(
        filename_base="YZD320_1.67Kpan1",
        start_T=-2, stop_T= 2,
        bidirectional=True,
        tg_bg_ratio = tg_bg_ratio,
        Vbg=Vbg, Vtg=Vtg, 
        waveplate_angles_deg=[63.7, 18.7],
        return_angle_deg=63.7,
        lf6_exp_time='100', lf6_frames='10', lf6_center='760',  # optional LF6 setupB
    )

variable: ['Vbg']
start: [0.]
end: [-18.45]
steps: 185
variable: ['Vtg']
start: [0.]
end: [27.95]
steps: 280
Exposetime(ms): 100.0
Frame_to_combine:10
Center Wave Length: 760.0
Grating: [750nm,600][2][0]
LF6 configured: {'exp_time': '100', 'frame_to_combine': '10', 'center': '760'}
[23:59:01] Pre-positioning to -2.0000 T (gate ±0.0010 T) …
[23:59:06] Reached start gate: B≈-2.0000 T
[23:59:07] Start ramping to +2.0000 T
[00:09:29] Stop ramping (forward end): B≈+2.0037 T
[00:09:30] Pre-positioning to +2.0000 T (gate ±0.0010 T) …
[00:09:56] Reached start gate: B≈+2.0006 T
[00:09:56] Start backward ramping to -2.0000 T
[00:20:09] Stop ramping (backward end): B≈-1.9994 T
Saved: YZD320\spectra\20251120_235900_YZD320_1.67Kpan1_Vbg=-18.45_Vtg=27.95V_E=45.002_D=8.102_760nmc_B-2.0Tto+2.0T_bi_inhalf63.7_18.7.csv
variable: ['Vbg']
start: [-18.45]
end: [-15.91]
steps: 26
variable: ['Vtg']
start: [27.95]
end: [25.35]
steps: 26
Exposetime(ms): 100.0
Frame_to_combine:10
Center Wave Length: 760.0
Grati

In [160]:
start_T=-2
stop_T=2

B_points = np.linspace(start_T,stop_T,11)
print(B_points)

[-2.  -1.6 -1.2 -0.8 -0.4  0.   0.4  0.8  1.2  1.6  2. ]


In [ ]:
# Vbgs = [-18.45, -15.91]
# Vtgs = [27.95, 25.35]

Vbgs=[-8.28]
Vtgs=[17.61]
tg_bg_ratio = 0.95

start_T=-2
stop_T=2

B_points = np.linspace(start_T,stop_T,41)

for Vbg,Vtg in zip(Vbgs,Vtgs):
    csv_path = sweep_fixed_gate_continuous_to_single_csv(
        filename_base="YZD320_1.67Kp2n1",
        start_T=start_T,          
        stop_T=stop_T,
        bidirectional=True,    
        tg_bg_ratio=tg_bg_ratio,
        Vbg=Vbg, Vtg=Vtg,
        waveplate_angles_deg=[75, 30],   # e.g. σ+ / σ−
        lf6_exp_time='200', lf6_frames='10', lf6_center='620',
        B_points=B_points,
    )

variable: ['Vbg']
start: [0.]
end: [-8.28]
steps: 83
variable: ['Vtg']
start: [0.]
end: [17.61]
steps: 177
Exposetime(ms): 200.0
Frame_to_combine:10
Center Wave Length: 620.0
Grating: [750nm,600][2][0]
LF6 configured: {'exp_time': '200', 'frame_to_combine': '10', 'center': '620'}
[12:47:55] Pre-positioning to -2.0000 T (gate ±0.0010 T) …
[12:48:00] Reached start gate: B≈-1.9995 T
[12:48:00] Set field to -2.0000 T (B point 1/41)
[12:48:14] Set field to -1.9000 T (B point 2/41)
[12:49:15] Set field to -1.8000 T (B point 3/41)
[12:50:32] Set field to -1.7000 T (B point 4/41)
[12:51:28] Set field to -1.6000 T (B point 5/41)
[12:52:43] Set field to -1.5000 T (B point 6/41)
[12:54:01] Set field to -1.4000 T (B point 7/41)
[12:54:57] Set field to -1.3000 T (B point 8/41)
[12:56:11] Set field to -1.2000 T (B point 9/41)
[12:57:27] Set field to -1.1000 T (B point 10/41)
[12:58:42] Set field to -1.0000 T (B point 11/41)
[12:59:57] Set field to -0.9000 T (B point 12/41)
[13:01:12] Set field to -0

In [428]:
wp_move_to(63.7
           )

In [ ]:
# Take Background file

paths = save_R0_for_angles(
    base_dir=rf"{SPECTRA_DIR}/Background",
    filename_base="YZD320_1.67Kp2n1",
    angles_deg=[71, 26],        # measure both angles
    Vbg=12.5, Vtg=13.16,           # set gate voltages for background
    gate_T=0.001,               # ±1 mT window for 0 T
    repeats=9, average=True,    # one averaged R(λ) per angle
    lf6_exp_time='200', lf6_frames='100', lf6_center='760',
    timestamp=True
)
paths

variable: ['Vbg']
start: [12.5]
end: [12.5]
steps: 2
variable: ['Vtg']
start: [13.16]
end: [13.16]
steps: 2
Exposetime(ms): 200.0
Frame_to_combine:100
Center Wave Length: 760.0
Grating: [750nm,600][2][0]
LF6 configured: {'exp_time': '200', 'frame_to_combine': '100', 'center': '760'}
Saved R0: YZD320\spectra\Background\YZD320_1.67Kp2n1_B0T_ang85_c760nm_exp200x100_Vbg+12.500_Vtg+13.160_R0_20251102_171718.csv
Saved R0: YZD320\spectra\Background\YZD320_1.67Kp2n1_B0T_ang40_c760nm_exp200x100_Vbg+12.500_Vtg+13.160_R0_20251102_172042.csv


['YZD320\\spectra\\Background\\YZD320_1.67Kp2n1_B0T_ang85_c760nm_exp200x100_Vbg+12.500_Vtg+13.160_R0_20251102_171718.csv',
 'YZD320\\spectra\\Background\\YZD320_1.67Kp2n1_B0T_ang40_c760nm_exp200x100_Vbg+12.500_Vtg+13.160_R0_20251102_172042.csv']

In [63]:
import sympy as sp

tg_start, bg_start, tg_stop, bg_stop = sp.symbols('tg_start bg_start tg_stop bg_stop')
tg_bg_ratio=0.95
Efield =45
doping = 16.5
# 16.2
#n=0   0
#n=1/3 3.68
#n=2/3 6.356
#n=1   9.0
# two equations
## -10, 18.8
eq1 = sp.Eq(tg_bg_ratio*tg_start - bg_start, Efield)
eq2 = sp.Eq(tg_bg_ratio*tg_start + bg_start, doping)


# solve for tg, bg
sol1 = sp.solve((eq1, eq2), (tg_start, bg_start))
# print(sol1)
# print(sol2)
print({k: float(v) for k, v in sol1.items()})

{bg_start: -14.25, tg_start: 32.36842105263158}


In [384]:
wp_move_to(63.7+45)
# thorlab_move_to(48.5)

In [63]:
frame_to_combine='20'
exp_time = '100'
repeat_n = 1
repeat_n2 = 1       
# centers = ['760','700']
centers = ['650']
# rot_mount.move_to(197.6)
# in_halfs = [85,40
in_halfs = [158, 248]
for center in centers:

    configure_lf6(exp_time=exp_time, frame_to_combine=frame_to_combine, center=center, verbose=True)
    for in_half in in_halfs:
        wp_move_to(in_half)
        sample_name = f'$YZD300$~$1.67KB=8Tp7$~$RefIH={in_half}deg{center}nm{exp_time}msx{frame_to_combine}x{repeat_n}$~'

        exps = spectral_experiments.SpectralExperimentsCollections()

        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG+BG=0$', iv, lf6, vbg_start= -25, vbg_stop= 19.5,
        #                                 vtg_start= 26.32, vtg_stop= -20.53, frames=446, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)            
        # #single line for background

        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG+BG=8.68$', iv, lf6, vbg_start= 14.34, vbg_stop= -20.66,
        #                         vtg_start= -5.96, vtg_stop= 30.88, frames=351, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG+BG=16.5$', iv, lf6, vbg_start= 18.25, vbg_stop= -14.25,
        #                 vtg_start= -1.84, vtg_stop=32.37, frames=326, repeat= repeat_n, plot= False)
        #         # single line for background
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background

        exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start= -10, vbg_stop= 15,
                                vtg_start= -10, vtg_stop= 15, frames=126, repeat= repeat_n, plot= False)
        # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=40$', iv, lf6, vbg_start= -23, vbg_stop= -10,
        #                 vtg_start= 17.89, vtg_stop= 31.58, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=35$', iv, lf6, vbg_start= -20.5, vbg_stop= -7.5,
        #                         vtg_start= 15.26, vtg_stop= 28.95, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=30$', iv, lf6, vbg_start= -18, vbg_stop= -5,
        #                 vtg_start= 12.63, vtg_stop=26.32, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=25$', iv, lf6, vbg_start= -15.5, vbg_stop= -2.5,
        #                 vtg_start= 10, vtg_stop= 23.68, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=20$', iv, lf6, vbg_start= -13, vbg_stop= 0,
        #         vtg_start= 7.37, vtg_stop= 21.05, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=15$', iv, lf6, vbg_start= -10.5, vbg_stop= 2.5,
        #         vtg_start= 4.74, vtg_stop= 18.42, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=10$', iv, lf6, vbg_start= -8, vbg_stop= 5,
        #                 vtg_start= 2.11, vtg_stop= 15.79, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        # #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=5$', iv, lf6, vbg_start= -5.5, vbg_stop= 7.5,
        #         vtg_start= -0.53, vtg_stop= 13.16, frames=261, repeat= repeat_n, plot= False)
        # # single line for background
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)   
        #single line for background
        
        # exps.add_dual_gate_spectra_sweep(sample_name, '$0.95TG-BG=0$', iv, lf6, vbg_start= -3, vbg_stop= 12.5,
        #                 vtg_start= -3.16, vtg_stop= 13.16, frames=311, repeat= repeat_n, plot= False)
        # if center == '700':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                     vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '760':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                     vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)
        # elif center == '640':
        #     exps.add_dual_gate_spectra_sweep(sample_name, '$EWS2BG0.95TG-BG=45$', iv, lf6, vbg_start=-14.1, vbg_stop= -14.1,
        #                                     vtg_start=  32.53, vtg_stop=  32.53, frames= 5, repeat= repeat_n, plot= False)  
        #single line for background
        # # single line for background
        # exps.add_dual_gate_spectra_sweep(sample_name, '$HBG0.95TG-BG=0$', iv, lf6, vbg_start=-8.4, vbg_stop= -8.4,
        #                                 vtg_start= -8.84, vtg_stop= -8.84, frames= 5, repeat= repeat_n, plot= False)
        # exps.add_dual_gate_spectra_sweep(sample_name, '$EBG0.95TG-BG=0$', iv, lf6, vbg_start=12.5, vbg_stop= 12.5,
        #                                 vtg_start= 13.16, vtg_stop= 13.16, frames= 5, repeat= repeat_n, plot= False)


        exps.execute(repeat=repeat_n2)
        winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

Exposetime(ms): 100.0
Frame_to_combine:20
Center Wave Length: 650.0
Grating: [750nm,600][2][0]
LF6 configured: {'exp_time': '100', 'frame_to_combine': '20', 'center': '650'}
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
variable: ['Vbg']
start: [0.]
end: [-10]
steps: 101
variable: ['Vtg']
start: [0.]
end: [-10]
steps: 101
variable: ['Vbg', 'Vtg']
start: [-10 -10]
end: [15 15]
steps: 126
variable: ['Vbg']
start: [15.]
end: [0]
steps: 151
variable: ['Vtg']
start: [15.]
end: [0]
steps: 151
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
variable: ['Vbg']
start: [0.]
end: [-10]
steps: 101
variable: ['Vtg']
start: [0.]
end: [-10]
steps: 101
variable: ['Vbg', 'Vtg']
start: [-10 -10]
end: [15 15]
steps: 126
variable: ['Vbg']
start: [15.]
end: [0]
steps: 151
variable: ['Vtg']
start: [15.]
end: [0]
steps: 151


In [22]:
# Persistent-mode timings (tune for your cryostat)
PMODE = {
    "warm_timeout_s": 90,    # time allowed for the heater to warm the switch
    "cool_timeout_s": 180,   # time allowed for the switch to cool superconducting
    "post_heat_dwell_s": 5,  # extra wait after heater reports hot
    "post_cool_dwell_s": 10, # extra wait after switch cools
    "temp_gate_enable": True,
    "temp_gate_K_max": 6.0,  # require magnet temp <= this after cooling (adjust!)
}

In [ ]:
# Enumerate ramp-rate table robustly (handles 0-based or 1-based indexing)

# Optional: import the vendor exception for cleaner handling
try:
    from attocube_sdk.ACS import AttoException
except Exception:
    AttoException = Exception  # fall back

def list_ramp_rates(magnet, ch=0):
    # detect indexing: try index 0 first; if it's out-of-range, start at 1
    start_idx = 0
    try:
        magnet.getRampRate(ch, 0)
    except AttoException as e:
        if "VALUEOUTOFRANGE" in str(e).upper() or "TOOHIGH" in str(e).upper():
            start_idx = 1

    rows = []
    i = start_idx
    while True:
        try:
            rng, rate = magnet.getRampRate(ch, i)  # returns (range, rate[T/min])
            rows.append((i, float(rng), float(rate)))
            i += 1
        except AttoException as e:
            # stop when we run off the end of the table
            if "VALUEOUTOFRANGE" in str(e).upper():
                break
            raise  # unexpected error: re-raise

    print(f"Ramp-rate table for channel {ch} (index, range, rate [T/min]):")
    for i, rng, rate in rows:
        print(f"  idx={i:2d}  range={rng:.4g}  rate={rate:.6g} T/min")

    if not rows:
        raise RuntimeError("No ramp-rate entries found.")
    max_idx, max_rng, max_rate = max(rows, key=lambda r: r[2])
    print(f"\nMax reported: {max_rate} T/min (index={max_idx}, range={max_rng})")
    return rows, (max_idx, max_rng, max_rate)

def list_default_ramp_rates(magnet, ch=0):
    try:
        nd = int(magnet.getNumDefaultRampRates(ch))
    except Exception:
        print("getNumDefaultRampRates not available on this firmware.")
        return []
    rows = []
    # Try both 0-based and 1-based for defaults too
    for start in (0, 1):
        rows.clear()
        for i in range(start, start + nd + 2):  # a little extra to discover the end
            try:
                rng, rate = magnet.getDefaultRampRate(i, ch)  # note (index, channel)
                rows.append((i, float(rng), float(rate)))
            except AttoException as e:
                if "VALUEOUTOFRANGE" in str(e).upper():
                    break
                # ignore other errors and continue probing
                continue
        if rows:
            break
    print(f"Default ramp rates for channel {ch}:")
    for i, rng, rate in rows:
        print(f"  idx={i:2d}  range={rng:.4g}  rate={rate:.6g} T/min")
    return rows

_ = list_default_ramp_rates(magnet, CHANNEL)



Default ramp rates for channel 0:
  idx= 0  range=40  rate=0.0344 T/min
Ramp-rate table for channel 0 (index, range, rate [T/min]):
  idx= 0  range=40  rate=0.0344 T/min
  idx= 1  range=44.01  rate=0.0172 T/min
  idx= 2  range=45  rate=0.0172 T/min
  idx= 3  range=46  rate=0.0086 T/min

Max reported: 0.0344 T/min (index=0, range=40.0)


In [ ]:
# Measure actual field slope near the start of a small move
import time, statistics
from collections import deque

def measure_dBdt_T_per_s(B_target, window_s=3.0, poll_hz=10):
    dt = 1.0/poll_hz
    buf = deque(maxlen=int(window_s*poll_hz)+2)
    # start a short ramp
    magnet.setHSetPoint(CHANNEL, float(B_target))
    magnet.startFieldControl(CHANNEL)
    t0 = time.time()
    while time.time() - t0 < window_s:
        time.sleep(dt)
        buf.append((time.time(), float(magnet.getH(CHANNEL))))
    (t1, B1), (t2, B2) = buf[0], buf[-1]
    dBdt = (B2 - B1) / max(1e-6, (t2 - t1))
    return dBdt



Measured dB/dt ≈ 0.000000 T/s


In [20]:
dBdt_T_per_s = measure_dBdt_T_per_s(B_target=0.05)  # small move
print(f"Measured dB/dt ≈ {dBdt_T_per_s:.6f} T/s")

Measured dB/dt ≈ -0.004547 T/s


In [17]:
# Example:
B_target = 0.20  # Tesla
print(f"Ramping to {B_target} T using the device's current ramp rate …")
B_final = ramp_to_using_current_rate(B_target)
print("Stable at ~", B_final, "T")

Ramping to 0.2 T using the device's current ramp rate …
Stable at ~ 0.1998 T


In [ ]:
# Example sweep (edit SWEEP dict above as needed). Use with caution.
sweep(SWEEP["start_T"], SWEEP["stop_T"], SWEEP["step_T"], SWEEP["rate_T_per_min"], SWEEP["repeats"], settle_s=0.0)
